In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 3 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 calibration.
# - Keep search centred on the ACTUAL incumbent.
# - Use a conservative empirical trust region.
# - Compare EI, posterior mean and UCB.
# - Do NOT automatically expand if an acquisition hits boundary.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 10 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 10 selected:
# [0.36520441, 0.55540850, 0.43974572]
#
# Prior prediction:
# mean ≈ -0.011575
# std  ≈ 0.011441
#
# Actual:
# -0.01675233935196823
# ------------------------------------------------------------

week10_pred_mean = -0.011575031459667956
week10_pred_std = 0.011441108629506683
week10_actual = -0.01675233935196823

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Empirical local scale
# ------------------------------------------------------------

other_mask = (
    np.arange(len(X))
    != best_idx
)

distances_to_best = np.linalg.norm(
    X[other_mask] - best_x,
    axis=1
)

nearest_distance = (
    distances_to_best.min()
)

empirical_cap = min(
    1.25 * nearest_distance,
    0.06
)

print("\n================================")
print("EMPIRICAL LOCAL SCALE")
print("================================")

print("Nearest observed point:")
print(nearest_distance)

print("\nEmpirical cap:")
print(empirical_cap)


# ------------------------------------------------------------
# 6. Conservative ARD trust region
# ------------------------------------------------------------

trust_half_width = np.clip(
    0.18 * lengthscales,
    0.015,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("WEEK 11 TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(trust_half_width)

print("\nLower:")
print(lower)

print("\nUpper:")
print(upper)


# ------------------------------------------------------------
# 7. Dense trust-region candidates
# ------------------------------------------------------------

rng = np.random.default_rng(42)

candidates = rng.uniform(
    lower,
    upper,
    size=(350000, 3)
)

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 8. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. Highest posterior mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distances
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Boundary check
# ------------------------------------------------------------

def boundary_status(
    x,
    lower,
    upper,
    tol=0.001
):

    status = []

    for j in range(len(x)):

        if abs(
            x[j] - lower[j]
        ) <= tol:

            status.append(
                f"x{j+1}=LOWER"
            )

        elif abs(
            x[j] - upper[j]
        ) <= tol:

            status.append(
                f"x{j+1}=UPPER"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        candidates[ei_idx],
        lower,
        upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        candidates[mean_idx],
        lower,
        upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        boundary_status(
            candidates[idx],
            lower,
            upper
        )
    )

DATA
X shape: (25, 3)
Y shape: (25,)

Current best:
[0.332348 0.553987 0.420645] -> -0.00263018431711505

Y range:
min = -0.3989255131463011
max = -0.00263018431711505
std = 0.07617532869806055

WEEK 10 CALIBRATION CHECK
Predicted mean: -0.011575031459667956
Predicted std : 0.011441108629506683
Actual        : -0.01675233935196823

Prediction error:
-0.005177307892300274

Error / predicted std:
-0.4525180260021278

GP FIT

Fitted kernel:
1.8**2 * Matern(length_scale=[0.829, 1.31, 0.273], nu=2.5) + WhiteKernel(noise_level=0.013)

ARD lengthscales:
[0.8290153  1.30696748 0.27272821]

Normalised inverse-lengthscale sensitivity:
[0.21394874 0.13570864 0.65034262]

EMPIRICAL LOCAL SCALE
Nearest observed point:
0.038031357824826625

Empirical cap:
0.047539197281033285

WEEK 11 TRUST REGION
Centre:
[0.332348 0.553987 0.420645]

Half-widths:
[0.0475392 0.0475392 0.0475392]

Lower:
[0.2848088 0.5064478 0.3731058]

Upper:
[0.3798872 0.6015262 0.4681842]

Candidates after duplicate filtering:
346

In [3]:
# ============================================================
# FUNCTION 3 - WEEK 11 FINAL TIGHT CHECK
# ============================================================
#
# Week 10 calibration improved substantially (~ -0.45 sigma),
# but the wider Week 11 trust region again pushes every
# acquisition to the x1 upper boundary.
#
# No candidate currently predicts a mean better than the
# observed incumbent, and the previous move toward larger x1
# did not improve the best value.
#
# Therefore perform ONE tighter incumbent-centred search.
# Do not expand after this.
# ============================================================


# ------------------------------------------------------------
# 1. Tight trust region
# ------------------------------------------------------------

tight_half_width = np.minimum(
    trust_half_width * 0.5,
    0.025
)

tight_half_width = np.maximum(
    tight_half_width,
    0.012
)

tight_lower = np.maximum(
    0.0,
    best_x - tight_half_width
)

tight_upper = np.minimum(
    1.0,
    best_x + tight_half_width
)

print("================================")
print("TIGHT TRUST REGION")
print("================================")

print("Centre:")
print(best_x)

print("\nHalf-widths:")
print(tight_half_width)

print("\nLower:")
print(tight_lower)

print("\nUpper:")
print(tight_upper)


# ------------------------------------------------------------
# 2. Candidate search
# ------------------------------------------------------------

rng_tight = np.random.default_rng(123)

tight_candidates = rng_tight.uniform(
    tight_lower,
    tight_upper,
    size=(400000, 3)
)

distance, _ = tree.query(
    tight_candidates,
    k=1
)

tight_candidates = tight_candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(tight_candidates))


# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------

tight_mu, tight_sigma = gp.predict(
    tight_candidates,
    return_std=True
)


# ------------------------------------------------------------
# 4. EI
# ------------------------------------------------------------

tight_EI = expected_improvement(
    tight_mu,
    tight_sigma,
    best_y,
    xi=0.0
)

tight_ei_idx = np.argmax(
    tight_EI
)

print("\n================================")
print("TIGHT EI")
print("================================")

print(
    "candidate =",
    tight_candidates[tight_ei_idx]
)

print(
    "mean =",
    tight_mu[tight_ei_idx]
)

print(
    "std =",
    tight_sigma[tight_ei_idx]
)

print(
    "EI =",
    tight_EI[tight_ei_idx]
)


# ------------------------------------------------------------
# 5. Highest predicted mean
# ------------------------------------------------------------

tight_mean_idx = np.argmax(
    tight_mu
)

print("\n================================")
print("TIGHT HIGHEST MEAN")
print("================================")

print(
    "candidate =",
    tight_candidates[tight_mean_idx]
)

print(
    "mean =",
    tight_mu[tight_mean_idx]
)

print(
    "std =",
    tight_sigma[tight_mean_idx]
)


# ------------------------------------------------------------
# 6. UCB
# ------------------------------------------------------------

print("\n================================")
print("TIGHT UCB")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(
        tight_UCB
    )

    print(
        f"beta={beta}",
        "\n candidate =",
        tight_candidates[idx],
        "\n mean =",
        round(tight_mu[idx], 6),
        "\n std =",
        round(tight_sigma[idx], 6),
        "\n UCB =",
        round(tight_UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 7. Distance from actual incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        tight_candidates[tight_ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        tight_candidates[tight_mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(
        tight_UCB
    )

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            tight_candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 8. Boundary check
# ------------------------------------------------------------

print("\n================================")
print("BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    boundary_status(
        tight_candidates[tight_ei_idx],
        tight_lower,
        tight_upper
    )
)

print(
    "Highest mean:",
    boundary_status(
        tight_candidates[tight_mean_idx],
        tight_lower,
        tight_upper
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    tight_UCB = (
        tight_mu
        + beta * tight_sigma
    )

    idx = np.argmax(
        tight_UCB
    )

    print(
        f"UCB beta={beta}:",
        boundary_status(
            tight_candidates[idx],
            tight_lower,
            tight_upper
        )
    )

TIGHT TRUST REGION
Centre:
[0.332348 0.553987 0.420645]

Half-widths:
[0.0237696 0.0237696 0.0237696]

Lower:
[0.3085784 0.5302174 0.3968754]

Upper:
[0.3561176 0.5777566 0.4444146]

Candidates after duplicate filtering:
384146

TIGHT EI
candidate = [0.35600377 0.54465817 0.44426682]
mean = -0.013340183746695058
std = 0.009987175127383896
EI = 0.0007235830060953537

TIGHT HIGHEST MEAN
candidate = [0.35611225 0.53842937 0.4324585 ]
mean = -0.013167930748657325
std = 0.009741488944527935

TIGHT UCB

beta=0.05 
 candidate = [0.35609507 0.53895899 0.43311855] 
 mean = -0.013168 
 std = 0.009748 
 UCB = -0.012681 

beta=0.1 
 candidate = [0.35609507 0.53895899 0.43311855] 
 mean = -0.013168 
 std = 0.009748 
 UCB = -0.012193 

beta=0.25 
 candidate = [0.35608853 0.53885524 0.43386917] 
 mean = -0.013169 
 std = 0.009756 
 UCB = -0.01073 

beta=0.5 
 candidate = [0.35609572 0.53928478 0.43600293] 
 mean = -0.013181 
 std = 0.009785 
 UCB = -0.008289 

beta=1.0 
 candidate = [0.35607984 0.544

In [4]:
# ============================================================
# FINAL FUNCTION 3 - WEEK 11 SELECTION
# ============================================================
#
# Week 10 calibration improved substantially (~ -0.45 sigma).
#
# Both the original and contracted Week 11 trust regions show
# a persistent preference for increasing x1.
#
# We do NOT expand further because previous movement in that
# direction has not beaten the actual incumbent.
#
# Within the tight region, choose the highest posterior mean.
# Low-beta UCB gives essentially the same answer, providing
# useful agreement without rewarding extra uncertainty.

final_idx = np.argmax(tight_mu)

week11_candidate = tight_candidates[final_idx]

print("Week 11 Function 3 candidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(tight_mu[final_idx])

print("\nPredicted std:")
print(tight_sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

Week 11 Function 3 candidate:
[0.35611225 0.53842937 0.4324585 ]

Predicted mean:
-0.013167930748657325

Predicted std:
0.009741488944527935

Distance from current best:
0.03076261767440085

Portal format:
0.356112-0.538429-0.432459
